In [37]:
# --- Ensure working directory is project root (contains 'parameter/') ---
import os
if not os.path.isdir('parameter') and os.path.isdir('../parameter'):
    os.chdir('..')


In [38]:
import numpy as np
import json


In [39]:
def create_safe_initial_conditions(N_AGENTS, V_CONST, critical_distance, desired_distance,
                                   margin, class_k1, class_k2,
                                   position_range=(-50, 50), max_attempts=10000):
    """
    Generate a random initial state matrix such that every CBF method
    (VO, FVO, HO) is feasible at t=0.
    """
    for attempt in range(max_attempts):
        initial_states_matrix = np.zeros((5, N_AGENTS))

        px = np.random.uniform(position_range[0], position_range[1], size=N_AGENTS)
        py = np.random.uniform(position_range[0], position_range[1], size=N_AGENTS)
        theta = np.random.uniform(0, 2 * np.pi, size=N_AGENTS)

        initial_states_matrix[0, :] = px
        initial_states_matrix[1, :] = py
        initial_states_matrix[4, :] = theta
        initial_states_matrix[2, :] = V_CONST * np.cos(theta)
        initial_states_matrix[3, :] = V_CONST * np.sin(theta)

        all_pairs_safe = True
        for i in range(N_AGENTS):
            for j in range(i + 1, N_AGENTS):
                p_i = initial_states_matrix[0:2, i]
                p_j = initial_states_matrix[0:2, j]
                v_i = initial_states_matrix[2:4, i]
                v_j = initial_states_matrix[2:4, j]

                p_ij = p_i - p_j
                v_ij = v_i - v_j

                pij_norm = np.linalg.norm(p_ij)
                vij_norm = np.linalg.norm(v_ij)
                pv_dot = np.dot(p_ij, v_ij)

                if pij_norm <= critical_distance * 1.5:
                    all_pairs_safe = False
                    break

                # pij_norm > 1.5 * critical implies disc > 0 and h_ds > 0 automatically,
                # so we only compute disc / sqrt_term for VO / FVO expressions below.
                disc = pij_norm**2 - critical_distance**2
                sqrt_term = np.sqrt(disc)

                h_vo = pv_dot + vij_norm * sqrt_term - margin
                if h_vo < 0:
                    all_pairs_safe = False
                    break

                if pij_norm * vij_norm <= 2 * V_CONST * sqrt_term:
                    h_fvo = (pv_dot
                             + (pij_norm / (4 * V_CONST)) * vij_norm**2
                             + V_CONST * pij_norm
                             - V_CONST * critical_distance**2 / pij_norm
                             - margin)
                else:
                    h_fvo = h_vo
                if h_fvo < 0:
                    all_pairs_safe = False
                    break

                h1 = disc
                psi1 = 2 * pv_dot + class_k1 * h1   # psi_1 = L_f h + alpha_1 * h
                if psi1 < 0:
                    all_pairs_safe = False
                    break


            if not all_pairs_safe:
                break

        if all_pairs_safe:
            return initial_states_matrix

    raise RuntimeError(f"Could not find safe initial conditions after {max_attempts} attempts.")


In [40]:
# --- Shared parameters (identical across all cases) ---
V_CONST = 20.0
STATE_DIM = 5

# CBF parameters (used by create_safe_initial_conditions)
class_k1 = 0.1
class_k2 = 0.1
margin = 1e-4
k_vo = 1000.0

# Number of initial conditions per case
test_case_num = 100


In [41]:
np.save('parameter/V_CONST.npy', V_CONST)
np.save('parameter/STATE_DIM.npy', STATE_DIM)
np.save('parameter/class_k1.npy', class_k1)
np.save('parameter/class_k2.npy', class_k2)
np.save('parameter/margin.npy', margin)
np.save('parameter/k_vo.npy', k_vo)
np.save('parameter/test_case_num.npy', test_case_num)


In [42]:
# --- Group A: Agent count sweep (desired=50, critical=40) ---
# Cases: (case_id, N_AGENTS, desired_distance, critical_distance)
CASES = [
    ('case0_N5', 5, 60, 40),
    ('case1_N10', 10, 60, 40),
    ('case2_N15', 15, 60, 40),
    ('case3_N20', 20, 60, 40),
    ('case4_N25', 25, 60, 40),
]

# CASES = [
#     ('case6_N35', 35, 60, 40),
#     ('case7_N40', 40, 60, 40),
#     ('case8_N45', 45, 60, 40),
#     ('case9_N50', 50, 60, 40),
# ]

# CASES = [
#      ('case0_N15', 15, 60, 40),
# ]

# CASES = [
#     ('case0_N5', 5, 60, 40),
#     ('case1_N10', 10, 60, 40),
#     ('case2_N15', 15, 60, 40),
#     ('case3_N20', 20, 60, 40),
# ]

# Base retry parameters
max_widen = 10
widen_factor = 1.2

cases_manifest = []
for case_id, N_AGENTS, desired_distance, critical_distance in CASES:
    print(f'\n==== {case_id} (N={N_AGENTS}, desired={desired_distance}, critical={critical_distance}) ====')

    # Per-case parameter save
    os.makedirs(f'parameter/{case_id}', exist_ok=True)
    np.save(f'parameter/{case_id}/N_AGENTS.npy', N_AGENTS)
    np.save(f'parameter/{case_id}/desired_distance.npy', desired_distance)
    np.save(f'parameter/{case_id}/critical_distance.npy', critical_distance)

    # Adaptive base_range — wider for larger N since HO constraint tightens
    if N_AGENTS >= 20:
        base_range = 1000.0
    elif N_AGENTS >= 15:
        base_range = 500.0
    else:
        base_range = 500.0
    print(f'  [{case_id}] base_range = ±{base_range:.0f}')
    # Generate initial conditions
    initial_test_case = np.zeros((STATE_DIM, N_AGENTS, test_case_num))
    for tc in range(test_case_num):
        current_range = base_range
        for widen_iter in range(max_widen + 1):
            try:
                initial_test_case[:, :, tc] = create_safe_initial_conditions(
                    N_AGENTS, V_CONST, critical_distance, desired_distance,
                    margin, class_k1, class_k2,
                    position_range=(-current_range, current_range),
                    max_attempts=10000,
                )
                break
            except RuntimeError:
                if widen_iter < max_widen:
                    current_range *= widen_factor
                    print(f'  [{case_id}] tc={tc}: widening range to ±{current_range:.0f}')
                else:
                    raise RuntimeError(f'{case_id}: tc={tc} infeasible after {max_widen} widenings')
        if (tc + 1) % 10 == 0:
            print(f'  [{case_id}] {tc+1}/{test_case_num} generated')

    # Save initial conditions
    os.makedirs(f'initial_conditions/{case_id}', exist_ok=True)
    np.save(f'initial_conditions/{case_id}/initial.npy', initial_test_case)
    cases_manifest.append({'case_id': case_id, 'N_AGENTS': N_AGENTS, 'desired_distance': desired_distance, 'critical_distance': critical_distance})
    print(f'  -> initial_conditions/{case_id}/initial.npy saved (shape={initial_test_case.shape})')

# Write cases manifest for run_all.sh to consume
with open('cases.json', 'w') as f:
    json.dump(cases_manifest, f, indent=2)
print('\nAll cases generated and cases.json written.')



==== case0_N5 (N=5, desired=60, critical=40) ====
  [case0_N5] base_range = ±500
  [case0_N5] 10/100 generated
  [case0_N5] 20/100 generated
  [case0_N5] 30/100 generated
  [case0_N5] 40/100 generated
  [case0_N5] 50/100 generated
  [case0_N5] 60/100 generated
  [case0_N5] 70/100 generated
  [case0_N5] 80/100 generated
  [case0_N5] 90/100 generated
  [case0_N5] 100/100 generated
  -> initial_conditions/case0_N5/initial.npy saved (shape=(5, 5, 100))

==== case1_N10 (N=10, desired=60, critical=40) ====
  [case1_N10] base_range = ±500
  [case1_N10] 10/100 generated
  [case1_N10] 20/100 generated
  [case1_N10] 30/100 generated
  [case1_N10] 40/100 generated
  [case1_N10] 50/100 generated
  [case1_N10] 60/100 generated
  [case1_N10] 70/100 generated
  [case1_N10] 80/100 generated
  [case1_N10] 90/100 generated
  [case1_N10] 100/100 generated
  -> initial_conditions/case1_N10/initial.npy saved (shape=(5, 10, 100))

==== case2_N15 (N=15, desired=60, critical=40) ====
  [case2_N15] base_range